# 05 — Rounds 2, 3, and 4 hyperparameter tuning

**Summary:** Three sequential sweeps in one notebook. Each round reads the **previous round’s results CSV** and fixes hyperparameters for the next sweep:

- **Default (`USE_AUTO_WINNER_FROM_PRIOR_ROUND = False`):** You set **`MANUAL_BASE_AFTER_ROUND1`**, **`MANUAL_BASE_AFTER_ROUND2`**, and **`MANUAL_BASE_AFTER_ROUND3`** (copy field-for-field from the `round1` / `round2` / `round3` CSV row you want to continue from) plus **`MANUAL_MAX_SEQ_LENGTH`** for all rounds.
- **Optional auto chain (`USE_AUTO_WINNER_FROM_PRIOR_ROUND = True`):** After each round, the next base config is **`pick_winner_by_eval_loss`** on that round’s results (lowest `eval_loss`).

**Sweeps:** (1) **Round 2:** `lora_r` and `lora_alpha = 2 * lora_r` — (2) **Round 3:** `lora_dropout` — (3) **Round 4:** batch / accumulation (optional: `TARGET_EFFECTIVE_BATCH`).

**Prerequisites:** Notebooks **03** (splits) and **04** (`round1_results.csv`).

**Recommended run order:** 03 → 04 → 05 → **08** (holdout eval for round1–4) → 06 → 07 → **09** (post-train holdout eval) → 10–12.

**Outputs:** `round2_results.csv`, `round3_results.csv`, `round4_results.csv` under `<WORKFLOW_ROOT>/lora_tuning_workflow/` (set `RUN_PROFILE_ID` to match notebook **03**).

#### Colab: mount (optional)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### Install

In [2]:
!pip -q install transformers peft accelerate bitsandbytes datasets trl cairosvg pillow scikit-learn lxml pandas

#### Imports

In [17]:
from src.core.runtime import cleanup_memory, set_seed
from src.training.lora.experiments import run_single_experiment_eval_loss_early_stop
from src.training.lora.tuning_utils import append_round_results_csv

In [3]:
import sys
from pathlib import Path

import pandas as pd
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

RAW_DIR = PROJECT_DIR / 'data' / 'raw'
INTERIM_DIR = PROJECT_DIR / 'data' / 'interim'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
RUN_PROFILE_ID = 'run_1' #'default'
WORKFLOW_ROOT = PROJECT_DIR / 'outputs' / 'workflow_runs' / RUN_PROFILE_ID
MODELS_ROOT = WORKFLOW_ROOT / 'models'
EXPERIMENT_ROOT = WORKFLOW_ROOT / 'lora_tuning_workflow'
ROUND1_PATH = EXPERIMENT_ROOT / 'round1_results.csv'
ROUND2_PATH = EXPERIMENT_ROOT / 'round2_results.csv'
ROUND3_PATH = EXPERIMENT_ROOT / 'round3_results.csv'
ROUND4_PATH = EXPERIMENT_ROOT / 'round4_results.csv'

print('WORKFLOW_ROOT:', WORKFLOW_ROOT)
print('cuda:', torch.cuda.is_available())

WORKFLOW_ROOT: /content/drive/MyDrive/DL_Midterm_Spring_2026_2/svg_project_DL/outputs/workflow_runs/run_1
cuda: True


#### Parameters

In [4]:
SEED = 42
VAL_FRAC = 0.10
TRAIN_FIRST_N = 7500 #None  # first N rows of train split; None = full train (shuffled split)

MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'

# False (default): use MANUAL_* dicts below for bases between rounds. True: pick_winner_by_eval_loss after each round.
USE_AUTO_WINNER_FROM_PRIOR_ROUND = False
# Used when USE_AUTO_WINNER_FROM_PRIOR_ROUND is False (fixed across rounds 2–4).
MANUAL_MAX_SEQ_LENGTH = 1536
# Paste from round1_results.csv / round2_results.csv / round3_results.csv rows you want as the starting point for the next round.
MANUAL_BASE_AFTER_ROUND1 = {
    'learning_rate': 2e-4,
    'max_steps': 125,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'early_stopping_patience': 3,
    'early_stopping_min_delta': 0.0,
    'eval_steps': 40,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 8,
}
MANUAL_BASE_AFTER_ROUND2 = {
    'learning_rate': 2e-4,
    'max_steps': 100,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'early_stopping_patience': 3,
    'early_stopping_min_delta': 0.0,
    'eval_steps': 40,
    'lora_r': 64,
    'lora_alpha': 128,
    'lora_dropout': 0.05,
    'per_device_train_batch_size': 4,
    'gradient_accumulation_steps': 8,
}
MANUAL_BASE_AFTER_ROUND3 = {
    'learning_rate': 2e-4,
    'max_steps': 50,
    'warmup_ratio': 0.05,
    'lr_scheduler_type': 'cosine',
    'early_stopping_patience': 3,
    'early_stopping_min_delta': 0.0,
    'eval_steps': 25,
    'lora_r': 64,
    'lora_alpha': 128,
    'lora_dropout': 0.05,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 8,
}

ROUND2_LORA_R_GRID = [16, 32, 64]
ROUND3_DROPOUT_GRID = [0.05, 0.10]
# (per_device_batch, grad_accum) pairs; set TARGET_EFFECTIVE_BATCH to None to disable check
ROUND4_BATCH_GRID = [(4, 8), (2, 8), (4,4)]
TARGET_EFFECTIVE_BATCH = None  # set to None to try every pair in ROUND4_BATCH_GRID

#### Load pool + previous results, rebuild train/val

In [13]:
from src.core.dataframe import choose_first_existing, train_val_split_df
from src.core.modeling_splits import load_train_val_pool
from src.training.lora.tuning_utils import pick_winner_by_eval_loss

train_val_pool = load_train_val_pool(WORKFLOW_ROOT)
PROMPT_COL = choose_first_existing(train_val_pool, ['prompt', 'description', 'text'], 'pool')
SVG_COL = choose_first_existing(train_val_pool, ['svg', 'svg_code', 'target', 'label'], 'pool')
_shuffle = TRAIN_FIRST_N is None
train_df, val_df = train_val_split_df(
    train_val_pool, val_frac=VAL_FRAC, seed=SEED, shuffle=_shuffle
)
if TRAIN_FIRST_N is not None:
    train_df = train_df.iloc[: int(TRAIN_FIRST_N)].reset_index(drop=True)
    n_val = min(len(val_df), max(1, int(VAL_FRAC * TRAIN_FIRST_N)))
    val_df = val_df.sample(n=n_val, random_state=SEED).reset_index(drop=True)

r1 = pd.read_csv(ROUND1_PATH)
if USE_AUTO_WINNER_FROM_PRIOR_ROUND:
    w1 = pick_winner_by_eval_loss(r1)
    print('Round 1 winner eval_loss:', w1['eval_loss'])
    base = {
        'learning_rate': float(w1['learning_rate']),
        'max_steps': int(w1['max_steps']),
        'warmup_ratio': float(w1.get('warmup_ratio', 0.05)),
        'lr_scheduler_type': str(w1.get('lr_scheduler_type', 'cosine')),
        'early_stopping_patience': int(w1.get('early_stopping_patience', 3)),
        'early_stopping_min_delta': float(w1.get('early_stopping_min_delta', 0.0)),
        'eval_steps': int(w1.get('eval_steps', 40)),
        'lora_r': int(w1['lora_r']),
        'lora_alpha': int(w1['lora_alpha']),
        'lora_dropout': float(w1['lora_dropout']),
        'per_device_train_batch_size': int(w1['per_device_train_batch_size']),
        'gradient_accumulation_steps': int(w1['gradient_accumulation_steps']),
    }
    max_seq_length = int(w1['max_seq_length'])
else:
    base = {k: MANUAL_BASE_AFTER_ROUND1[k] for k in MANUAL_BASE_AFTER_ROUND1}
    max_seq_length = int(MANUAL_MAX_SEQ_LENGTH)
    print('Manual base for round 2 (from MANUAL_BASE_AFTER_ROUND1):', max_seq_length, base)
print('Fixed max_seq_length:', max_seq_length, base)

Manual base for round 2 (from MANUAL_BASE_AFTER_ROUND1): 1536 {'learning_rate': 0.0002, 'max_steps': 125, 'warmup_ratio': 0.05, 'lr_scheduler_type': 'cosine', 'early_stopping_patience': 3, 'early_stopping_min_delta': 0.0, 'eval_steps': 40, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 8}
Fixed max_seq_length: 1536 {'learning_rate': 0.0002, 'max_steps': 125, 'warmup_ratio': 0.05, 'lr_scheduler_type': 'cosine', 'early_stopping_patience': 3, 'early_stopping_min_delta': 0.0, 'eval_steps': 40, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 8}


#### Round 2 — sweep LoRA rank

In [6]:

set_seed(SEED)
cleanup_memory()
if ROUND2_PATH.exists():
    ROUND2_PATH.unlink()

for i, r in enumerate(ROUND2_LORA_R_GRID, start=1):
    cfg = {**base, 'lora_r': int(r), 'lora_alpha': int(2 * r)}
    run_name = f'r2_{i:03d}_r{r}_seed{SEED}'
    summary, _, run_dir, reg_id = run_single_experiment_eval_loss_early_stop(
        run_name=run_name,
        config=cfg,
        train_df=train_df,
        val_df=val_df,
        prompt_col=PROMPT_COL,
        svg_col=SVG_COL,
        model_id=MODEL_ID,
        max_seq_length=max_seq_length,
        root_dir=EXPERIMENT_ROOT,
        project_root=PROJECT_DIR,
        tuning_stage='round2',
        curriculum=False,
        seed=SEED,
        eval_steps=cfg.get('eval_steps'),
        notes='round2 lora_r',
        models_root=MODELS_ROOT,
    )
    row = {**summary, 'max_seq_length': max_seq_length, 'tuning_stage': 'round2'}
    append_round_results_csv(ROUND2_PATH, row)
    cleanup_memory()

r2 = pd.read_csv(ROUND2_PATH)
if USE_AUTO_WINNER_FROM_PRIOR_ROUND:
    w2 = pick_winner_by_eval_loss(r2)
    base2 = {
        'learning_rate': float(w2['learning_rate']),
        'max_steps': int(w2['max_steps']),
        'warmup_ratio': float(w2.get('warmup_ratio', 0.05)),
        'lr_scheduler_type': str(w2.get('lr_scheduler_type', 'cosine')),
        'early_stopping_patience': int(w2.get('early_stopping_patience', 3)),
        'early_stopping_min_delta': float(w2.get('early_stopping_min_delta', 0.0)),
        'eval_steps': int(w2.get('eval_steps', 40)),
        'lora_r': int(w2['lora_r']),
        'lora_alpha': int(w2['lora_alpha']),
        'lora_dropout': float(w2['lora_dropout']),
        'per_device_train_batch_size': int(w2['per_device_train_batch_size']),
        'gradient_accumulation_steps': int(w2['gradient_accumulation_steps']),
    }
    print('Round 2 winner:', base2)
else:
    base2 = {k: MANUAL_BASE_AFTER_ROUND2[k] for k in MANUAL_BASE_AFTER_ROUND2}
    print('Manual base for round 3 (MANUAL_BASE_AFTER_ROUND2):', base2)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[device] torch.cuda.is_available()=True
[device] cuda:0 = NVIDIA A100-SXM4-40GB
[device] first trainable param device=cuda:0


Adding EOS to train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/4990 [00:00<?, ? examples/s]

...

Tokenizing train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/4990 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4990 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
40,0.429497,0.414367
80,0.419371,0.383520
120,0.388600,0.377050


Manual base for round 3 (MANUAL_BASE_AFTER_ROUND2): {'learning_rate': 0.0002, 'max_steps': 125, 'warmup_ratio': 0.05, 'lr_scheduler_type': 'cosine', 'early_stopping_patience': 3, 'early_stopping_min_delta': 0.0, 'eval_steps': 40, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 8}


#### Round 3 — sweep dropout

In [11]:
set_seed(SEED)
cleanup_memory()
if ROUND3_PATH.exists():
    ROUND3_PATH.unlink()

for i, do in enumerate(ROUND3_DROPOUT_GRID, start=1):
    cfg = {**base2, 'lora_dropout': float(do)}
    run_name = f'r3_{i:03d}_do{do}_seed{SEED}'
    summary, _, run_dir, reg_id = run_single_experiment_eval_loss_early_stop(
        run_name=run_name,
        config=cfg,
        train_df=train_df,
        val_df=val_df,
        prompt_col=PROMPT_COL,
        svg_col=SVG_COL,
        model_id=MODEL_ID,
        max_seq_length=max_seq_length,
        root_dir=EXPERIMENT_ROOT,
        project_root=PROJECT_DIR,
        tuning_stage='round3',
        curriculum=False,
        seed=SEED,
        eval_steps=cfg.get('eval_steps'),
        notes='round3 dropout',
        models_root=MODELS_ROOT,
    )
    row = {**summary, 'max_seq_length': max_seq_length, 'tuning_stage': 'round3'}
    append_round_results_csv(ROUND3_PATH, row)
    cleanup_memory()

r3 = pd.read_csv(ROUND3_PATH)
if USE_AUTO_WINNER_FROM_PRIOR_ROUND:
    w3 = pick_winner_by_eval_loss(r3)
    base3 = {
        'learning_rate': float(w3['learning_rate']),
        'max_steps': int(w3['max_steps']),
        'warmup_ratio': float(w3.get('warmup_ratio', 0.05)),
        'lr_scheduler_type': str(w3.get('lr_scheduler_type', 'cosine')),
        'early_stopping_patience': int(w3.get('early_stopping_patience', 3)),
        'early_stopping_min_delta': float(w3.get('early_stopping_min_delta', 0.0)),
        'eval_steps': int(w3.get('eval_steps', 40)),
        'lora_r': int(w3['lora_r']),
        'lora_alpha': int(w3['lora_alpha']),
        'lora_dropout': float(w3['lora_dropout']),
        'per_device_train_batch_size': int(w3['per_device_train_batch_size']),
        'gradient_accumulation_steps': int(w3['gradient_accumulation_steps']),
    }
    print('Round 3 winner:', base3)
else:
    base3 = {k: MANUAL_BASE_AFTER_ROUND3[k] for k in MANUAL_BASE_AFTER_ROUND3}
    print('Manual base for round 4 (MANUAL_BASE_AFTER_ROUND3):', base3)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[device] torch.cuda.is_available()=True
[device] cuda:0 = NVIDIA A100-SXM4-40GB
[device] first trainable param device=cuda:0


Adding EOS to train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

...

Tokenizing train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
40,0.456869,0.451759
80,0.447579,0.422430
120,0.418003,0.416111


Manual base for round 4 (MANUAL_BASE_AFTER_ROUND3): {'learning_rate': 0.0002, 'max_steps': 100, 'warmup_ratio': 0.05, 'lr_scheduler_type': 'cosine', 'early_stopping_patience': 3, 'early_stopping_min_delta': 0.0, 'eval_steps': 40, 'lora_r': 64, 'lora_alpha': 128, 'lora_dropout': 0.05, 'per_device_train_batch_size': 1, 'gradient_accumulation_steps': 8}


#### Round 4 — sweep batch / accumulation

In [18]:
set_seed(SEED)
cleanup_memory()
if ROUND4_PATH.exists():
    ROUND4_PATH.unlink()

j = 0
for pbs, gas in ROUND4_BATCH_GRID:
    if TARGET_EFFECTIVE_BATCH is not None and pbs * gas != TARGET_EFFECTIVE_BATCH:
        continue
    j += 1
    cfg = {**base3, 'per_device_train_batch_size': int(pbs), 'gradient_accumulation_steps': int(gas)}
    run_name = f'r4_{j:03d}_bs{pbs}_ga{gas}_seed{SEED}'
    summary, _, run_dir, reg_id = run_single_experiment_eval_loss_early_stop(
        run_name=run_name,
        config=cfg,
        train_df=train_df,
        val_df=val_df,
        prompt_col=PROMPT_COL,
        svg_col=SVG_COL,
        model_id=MODEL_ID,
        max_seq_length=max_seq_length,
        root_dir=EXPERIMENT_ROOT,
        project_root=PROJECT_DIR,
        tuning_stage='round4',
        curriculum=False,
        seed=SEED,
        eval_steps=cfg.get('eval_steps'),
        notes='round4 batch',
        models_root=MODELS_ROOT,
    )
    row = {**summary, 'max_seq_length': max_seq_length, 'tuning_stage': 'round4'}
    append_round_results_csv(ROUND4_PATH, row)
    cleanup_memory()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[device] torch.cuda.is_available()=True
[device] cuda:0 = NVIDIA A100-SXM4-40GB
[device] first trainable param device=cuda:0


Adding EOS to train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

...

Adding EOS to train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/7500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
25,0.463532,0.460336
50,0.425220,0.444218


In [19]:
r4 = pd.read_csv(ROUND4_PATH)
print(r4.sort_values('eval_loss').head())
print('Done rounds 2–4.')

                run_name  train_rows  val_rows  learning_rate  lora_r  \
0  r4_001_bs4_ga8_seed42        7500       750         0.0002      64   
1  r4_002_bs2_ga8_seed42        7500       750         0.0002      64   
2  r4_003_bs4_ga4_seed42        7500       750         0.0002      64   

   lora_alpha  lora_dropout  per_device_train_batch_size  \
0         128          0.05                            4   
1         128          0.05                            2   
2         128          0.05                            4   

   gradient_accumulation_steps  effective_batch_size  ...  \
0                            8                    32  ...   
1                            8                    16  ...   
2                            4                    16  ...   

                                best_loss_checkpoint  svg_open_rate  \
0  /content/drive/MyDrive/DL_Midterm_Spring_2026_...            NaN   
1  /content/drive/MyDrive/DL_Midterm_Spring_2026_...            NaN   
2  /cont

In [20]:
r4.sort_values('eval_loss').head()

,run_name,train_rows,val_rows,learning_rate,lora_r,lora_alpha,lora_dropout,per_device_train_batch_size,gradient_accumulation_steps,effective_batch_size,...,best_loss_checkpoint,svg_open_rate,svg_close_rate,xml_parse_rate,render_rate,avg_pred_char_len,submission_valid_rate,registry_model_id,max_seq_length,tuning_stage
0,r4_001_bs4_ga8_seed42,7500,750,0.0002,64,128,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,b90986cfef10424f,1536,round4
1,r4_002_bs2_ga8_seed42,7500,750,0.0002,64,128,0.05,2,8,16,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,eafb41fbcdc84ab7,1536,round4
2,r4_003_bs4_ga4_seed42,7500,750,0.0002,64,128,0.05,4,4,16,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,5d4370e7697140ba,1536,round4


In [21]:
r3.sort_values('eval_loss').head()

,run_name,train_rows,val_rows,learning_rate,lora_r,lora_alpha,lora_dropout,per_device_train_batch_size,gradient_accumulation_steps,effective_batch_size,...,best_loss_checkpoint,svg_open_rate,svg_close_rate,xml_parse_rate,render_rate,avg_pred_char_len,submission_valid_rate,registry_model_id,max_seq_length,tuning_stage
0,r3_001_do0.05_seed42,7500,750,0.0002,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,1c028d1d5dd34a1d,1536,round3
1,r3_002_do0.1_seed42,7500,750,0.0002,16,32,0.10,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,067768ef650c49da,1536,round3


In [24]:
r2.sort_values('eval_loss').head()

,run_name,train_rows,val_rows,learning_rate,lora_r,lora_alpha,lora_dropout,per_device_train_batch_size,gradient_accumulation_steps,effective_batch_size,...,best_loss_checkpoint,svg_open_rate,svg_close_rate,xml_parse_rate,render_rate,avg_pred_char_len,submission_valid_rate,registry_model_id,max_seq_length,tuning_stage
2,r2_003_r64_seed42,7500,4990,0.0002,64,128,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,895df98c4cff4892,1536,round2
1,r2_002_r32_seed42,7500,4990,0.0002,32,64,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,cd09b0b598b04c8e,1536,round2
0,r2_001_r16_seed42,7500,4990,0.0002,16,32,0.05,4,8,32,...,/content/drive/MyDrive/DL_Midterm_Spring_2026_...,NaN,NaN,NaN,NaN,NaN,NaN,e87a05edb1aa4da1,1536,round2
